# Master Setup Notebook

Run this notebook to create the complete folder structure and configuration files to run the Eutrophication Workbench workflow.


### Instructions

1. Execute the Python cell below to set up the directories and copy necessary files.
2. Enter the required inputs when prompted (user initials, years month, depths).
3. The script will create all folders and generate configuration files automatically.

> **Notes:**
> - Please **do not run this notebook in its original shared folder**.  
> - Before executing any cell, create a **copy in your Home workspace** (`/home/jovyan/workspace`).  
> - This ensures that all generated files are stored safely in your personal area and prevents conflicts with shared resources.


### Folder Structure Created

- `1_DatalakeQuery/`: For BEACON queries on merged BEACON instanse using jupyter notebooks
- `2_CWduplicates-tool/`: Configuration for duplicate removal tool and outputs
- `3_FileForge/`: FileForge configurations and outputs

---
### Workflow Overview

| Folder | Role | Execution Environment |
|------|-----|-------------|
| `1_BEACON` | Data access & harmonisation | JupyterLab |
| `2_CWduplicates-tool` | Duplicate detection | CCP |
| `3_FileForge` | Cleaning & format conversion | CCP |
| `4_webODV` | Exploration & QC | webODV |

---

###  Important notes
> - Ensure you have access to the source directories for copying files. If not, warnings will be printed.
> - The default time and depth range are 2010-01-01T00:00:00-2010-01-31T23:59:59 and 0-10m for the global ocean, bear in mind that this is also the default on your jupyter notebook `01_EWB_BEACON=merged-v1.5.4_FILTER=BDI-FeatureType_VAR=allEOV_OUT=parquet.ipynb` in **step 1**, thus if you change the default values you will need to change you notebook as well.
> - At the end of step 3, copy .txt files from `3_FileForge/outputs` to `/home/jovyan/<4_webODV>` to import it in webODV.
> - ⚠️ The EWB workflow is **hybrid**:
>   - Not all steps are automated
>   - Some steps require **user decisions**


### Access limitations
If you do not writen have access to `/blue-cloud-dataspace`, it means you are not part of the developer team.  
In that case, the you are running the workflow from your workspace and the **size of the datasets you can process will be limited**.

---

## Authorship

* **Author:** Nydia Catalina Reyes Suarez ([![ORCID](https://orcid.org/sites/default/files/images/orcid_16x16.png)](https://orcid.org/0000-0002-3906-471X) [ORCID](https://orcid.org/0000-0002-3906-471X), [Github](https://github.com/Geeokta))
* **Affiliation:** Istituto Nazionale di Oceanografia e di Geofisica Sperimentale - OGS  
* **Contact:** nreyessuarez@ogs.it  
* **Release date:** 2026-04-30

---

## License
This notebook is licensed under the [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/).

© 2026 NYDIA CATALINA REYES SUAREZ. All rights reserved.

---
## How to Cite:
Reyes Suarez, Nydia Catalina (2026): Jupyter Notebook - Eutrophication Workbench Master Setup User Workflow. v.1.0.0. DOI: 10.5281/zenodo.19923943. Retrieved from the Blue-Cloud Gateway (https://blue-cloud.d4science.org), Eutrophication Workbench VLab (https://blue-cloud.d4science.org/group/eutrophication-workbench) operated by D4Science.org (www.d4science.org - ROR https://ror.org/027cvs066) 

---


In [ ]:
from pathlib import Path
from datetime import datetime
from shutil import copy2, copyfile
import os

root = Path.cwd()
user = input("Enter your initials: ").strip().replace(" ", "_")
if not user:
    user = "anonymous"

year_from = input("Enter start year (YYYYMM): ").strip()
year_to = input("Enter end year (YYYYMM): ").strip()
depth_from = input("Enter depth from (m): ").strip()
depth_to = input("Enter depth to (m): ").strip()

if not year_from:
    year_from = "201001"
if not year_to:
    year_to = "20101"
if not depth_from:
    depth_from = "0"
if not depth_to:
    depth_to = "10"

date = datetime.now().strftime("%Y%m%d")
output_base = root / "1_DatalakeQuery" / "outputs" / f"{date}_{user}"

dirs = [
    root / "1_DatalakeQuery" / "Merged_notebooks",
    root / "2_CWduplicates-tool" / "Configuration_files" / "Config_Profiles",
    root / "2_CWduplicates-tool" / "outputs",
    root / "2_CWduplicates-tool" / "cache_empty",
    root / "3_FileForge"  / "Configuration_files",
    root / "3_FileForge"  / "outputs",
    
]

for path in dirs:
    path.mkdir(parents=True, exist_ok=True)
    
# Copy BEACON merged notebook
src_nb = Path(os.path.expanduser("~/workspace/VREFolders/Eutrophication-Workbench/1_Beacon/Merged_notebooks/01_EWB_BEACON=merged-v1.5.4_FILTER=BDI-FeatureType_VAR=allEOV_OUT=parquet.ipynb"))
dst_nb = root / "1_DatalakeQuery" / "Merged_notebooks" / src_nb.name
dst_nb.parent.mkdir(parents=True, exist_ok=True)

try:
    copyfile(src_nb, dst_nb)
except OSError as exc:
    print("Failed to copy notebook:", exc)
    raise

# Copy FileForge configuration files
src_config = Path(os.path.expanduser("~/workspace/VREFolders/Eutrophication-Workbench/3_FileForge/Configuration_files/"))
dst_config = root / "3_FileForge" / "Configuration_files"

if src_config.exists():
    for config_file in src_config.glob("*"):
        if config_file.is_file():
            dest_file = dst_config / config_file.name
            try:
                copyfile(config_file, dest_file)
            except OSError as exc:
                print(f"Failed to copy {config_file}:", exc)
                raise
else:
    print(f"Warning: Source config directory not found at {src_config}")

# Create the configuation file for CW (DO NOT modify it if you have not received the proper training, otherwise you might not be able to run the method)
cw_profile_template = f"""default :
  operator :
    name : "{user}"
  output :
    filename : "02_BC_EWB_CW_PR_{depth_from}m-{depth_to}m_{year_from}-{year_to}"
  #-----------------------------
  # SOURCE INFORMATION
  #-------------------------
  sources :
    - src_short_name : "BEACON_CMEMS_BGC"
      raw_data :
        flag : 1
        reader_type : "BEACON_PARQUET"
        data_dir : "EUTRO_CMEMS"
      trustlevel : 1

    - src_short_name : "BEACON_EC"
      raw_data :
        flag : 1
        reader_type : "BEACON_PARQUET"
        data_dir : "EUTRO_EC"
      trustlevel : 2

    - src_short_name : "BEACON_WOD"
      raw_data :
        flag : 1
        reader_type : "BEACON_PARQUET"
        data_dir : "EUTRO_WOD"
      trustlevel : 3

  #--------------------------
  #DATA SELECTION
  #------------------------------
  data_sel :
    - region : NULL
    - instr : NULL
    - groupParam : NULL
    - dataType : ["TRP", "PR", "TSP"]
    - period : NULL
  dupli_def :
    time_unit : "minutes"
    time_diff : 15
    pos_diff : 15
    instr_option : 2
    src_option : 2
    aggr_option : 4
    graph_flag : 1
  removal :
    flag : 1
    group : 0
    rules_to_apply : [2,3,4,5]
"""

profile_path = (root / "2_CWduplicates-tool" / "Configuration_files" / "Config_Profiles" / "cw_user_config.yaml")
profile_path.write_text(cw_profile_template, encoding="utf-8")


print("Created master folder structure under:", root)